In [ ]:
import numpy as np
import pandas as pd
import os
import s3fs
import sys
sys.path.append('../outils')

import fonctions_nomenclature as fnom
import fonctions_preparation_table as fprep
%load_ext autoreload
%autoreload 2

np.random.seed(123)

# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})
BUCKET = "mathildedonnenfeld/diffusion/bistrots_et_bulletins"

In [ ]:
FILE_KEY_S3 = "t1_muni.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    t1_muni = pd.read_csv(file_in, sep=",")

t1_muni.head()

In [ ]:
FILE_KEY_S3 = "t2_muni.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    t2_muni = pd.read_csv(file_in, sep=",")

t2_muni.head()

In [ ]:
FILE_KEY_S3 = "bistrots_sirene.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
 bistrots_sirene = pd.read_csv(file_in, sep=",", dtype={"codeCommuneEtablissement": str})

bistrots_sirene["codeCommuneEtablissement"] = bistrots_sirene["codeCommuneEtablissement"].str.strip().str.zfill(5)
bistrots_sirene.head()

In [ ]:
codes = bistrots_sirene["codeCommuneEtablissement"].dropna().astype(str)
print((codes.str.len() != 5).sum(), "codes de longueur différente de 5")
print(codes[codes.str.len() != 5].unique()[:10])

In [ ]:
FILE_KEY_S3 = "filosofi.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    filosofi = pd.read_csv(file_in, sep=",")

filosofi.head()

## Explo et prep données bistrots 

In [ ]:
bistrots_sirene.describe()

In [ ]:
# Data prep Sirene ----------------------------
## Modifie le format de la dateDernierTraitementEtablissement (enlever l'heure) et de la dateCreationEtablissement 
## Met à NA les dates de création au 1er janvier 1900 ou 1901
bistrots_sirene["date_dernier_traitement"] = pd.to_datetime(bistrots_sirene.dateDernierTraitementEtablissement).dt.normalize()
bistrots_sirene["date_creation"] = pd.to_datetime(bistrots_sirene.dateCreationEtablissement.replace("1900-01-01", np.nan).replace("1901-01-01", np.nan)).dt.normalize()
bistrots_sirene.head()

In [ ]:
bistrots_sirene["date_dernier_traitement"].describe()

In [ ]:
bistrots_sirene["date_creation"].describe()

In [ ]:
cond_date = "2026-01-01"
print(bistrots_sirene["siret"].nunique())
print(bistrots_sirene[(bistrots_sirene['date_creation'] <= cond_date)]["siret"].nunique())
print(bistrots_sirene[(bistrots_sirene['date_creation'].isnull())]["siret"].nunique())

In [ ]:
#Agrégation Paris-Lyon-Marseille.

arr = bistrots_sirene["codeCommuneEtablissement"]
bistrots_sirene["code_commune"] = arr.mask(arr.str.startswith("751"), "75056") \
                                     .mask(arr.str.match(r"^6938[1-9]$"), "69123") \
                                     .mask(arr.str.match(r"^132(0[1-9]|1[0-6])$"), "13055")

In [ ]:
#Paris
(bistrots_sirene[bistrots_sirene.libelleCommuneEtablissement.str.contains("PARIS")]["code_commune"]=='75056').describe()

In [ ]:
#Lyon
(bistrots_sirene[bistrots_sirene.libelleCommuneEtablissement.str.contains("LYON")]["code_commune"]=='69123').describe()

In [ ]:
#Marseille
(bistrots_sirene[bistrots_sirene.libelleCommuneEtablissement.str.contains("MARSEILLE")]["code_commune"]=='13055').describe()

In [ ]:
bistrots_par_ville = (
    bistrots_sirene
    .assign(
        age=(
            pd.Timestamp("2026-03-01")
            - pd.to_datetime(
                bistrots_sirene["date_creation"],
                errors="coerce"
            )
        ).dt.days / 365.25
    )
    .groupby(["code_commune"])
    .agg(
        nb_etablissements=("siret", "nunique"),
        age_moyen_bistrots=("age", "mean")
    )
    .reset_index()
    .sort_values("nb_etablissements", ascending=False)
)

bistrots_par_ville["nb_etablissements"] = bistrots_par_ville["nb_etablissements"].fillna(0)

In [ ]:
bistrots_par_ville.head(20)

## Explo et prep données municipales

On cherche à construire 4 indicateurs : 

1°) Le vote de droite : 2 si la nuance majoritaire au conseil municipal est sans étiquette, 1 si elle est de droite, 0 sinon 

2°) La majorité de droite : 2 si la couleur majoritaire au conseil municipal est sans étiquette, 1 si elle est de droite, 0 sinon (indépendamment de la liste arrivée en tête)*

3°) Le vote d'extrême droite : 2 si la nuance majoritaire au conseil municipal est sans étiquette, 1 si la nuance majoritaire au conseil municipal est d'extrême droite, 0 sinon

4°) Le vote "HorizAttal" : 2 si la nuance majoritaire au conseil municipal est sans étiquette, 1 si la nuance majoritaire est le parti "Horizon" (celui d'Edouard Philippe, néanmoins le plus proche sur le spectre politique de celui de Gabriel Attal), 0 sinon

*Depuis 2025, pour les communes de Paris, Lyon et Marseille, la prime majoritaire est abaissée à 25 % pour le conseil municipal (contre 50 % avant). Cette modification implique que la liste arrivée en tête ne détient pas nécessairement la majorité des sièges au conseil municipal. Les listes arrivées derrière elle peuvent obtenir, collectivement, plus de la moitié des sièges (par exemple, une liste de gauche peut arriver en tête, mais la majorité de sièges au conseil municipal être de droite si les listes suivantes sont de droite).
L'indicateur 2 est construit afin de vérifier que la couleur de la majorité des votes est cohérente avec la couleur liste arrivée en tête. Si tel est le cas, nous nous concentrerons sur les indicateurs 1, 3 et 4 dans l'analyse. 

### Préparation de la table du deuxième tour
On commence par traiter les villes ayant eu deux tours aux élections municipales

In [ ]:
t2_muni.head(20)

In [ ]:
t2_muni.describe()

In [ ]:
t2_muni.columns #=> trop de colonnes, il faut sélectionner les colonnes pertinentes 

In [ ]:
# Synthèse de la table du deuxième tour ------------------------
t2_muni_long = fprep.synthese_table_muni(t2_muni)

t2_muni_long.head()

### Préparation de la table du premier tour

Informations sur T1 : 
- La table du premier tour contient les communes ayant eu deux tours aux élections, donc des communes déjà traitées dans T2. Il s'agit donc de sélectionner négativement les communes non encore traitées. 
- La table T1 a la même structure que la table T2 : les fonctions utilisées pour T2 peuvent l'être pour T1

In [ ]:
# Selection des villes n'ayant eu qu'un seul tour d'élection (en pratique : on enlève celles qui en ont eu 2)

## Récupérer les codes communes des communes ayant eu deux tours d'élection 
cc_2t = t2_muni_long["Code commune"].unique()

## Mise au format des codes commune dans la table t1 : rajouter un 0 devant les codes commune avec 4 chiffres
t1_muni["Code commune"]= t1_muni["Code commune"].astype(str).str.zfill(5)
## Sélection inversée 
t1_muni_keep = t1_muni[~t1_muni["Code commune"].isin(cc_2t)] 

In [ ]:
# Vérification : a-t-on toutes les villes ? 
print(t1_muni["Code commune"].unique().shape[0]) # 34836
print(t1_muni_keep["Code commune"].unique().shape[0] +len(cc_2t)) #34836

In [ ]:
# Préparation de T1 à l'image de T2 
t1_muni_long = fprep.synthese_table_muni(t1_muni_keep)

## Vérif : je veux voir si le "nb de codes communes de T1 avant sélection (ie, ttes les communes)" == nb codes communes t1_keep + nb code commune t2, pour voir si mon code "synthese_table_muni" ne supprime pas des communes
print(len(t1_muni["Code commune"].unique()))
print(len(t1_muni_long["Code commune"].unique()) + len(cc_2t))


In [ ]:
t1_muni_long.columns

In [ ]:
# Création variable nombre de candidats => prend toutes les colonnes vides, il faut select celles ou CM est non vide ? 
t1_muni_long['Nombre candidats'] = (
    t1_muni_long['Sièges au CM'].notna()
    .groupby(t1_muni_long['Code commune'])
    .transform('sum')
)
t1_muni_long.describe()

In [ ]:
# Création % voix liste majo ----------------------------------
t1_muni_voix_majo = fprep.calcul_part_vote_liste_gagnante(t1_muni_long)
t1_muni_voix_majo[t1_muni_voix_majo['Code commune']=="27333"] # Vérification qu'on a ce qu'on veut pour Heudicourt-city

In [ ]:
# Création d'une nomenclature associant une couleur à une nuance ------------

## Récupérer les nuances
label_nuance = pd.DataFrame({"Nuance": pd.concat([t1_muni_long["Nuance"], t2_muni_long["Nuance"]]).unique()})
# print(label_nuance)

nomenclature_couleur_nuance = fnom.classer_nuance(label_nuance) 
#print(nomenclature_couleur_nuance)

In [ ]:
# Création variable nombre de candidats
t2_muni_long['Nombre candidats'] = (
    t2_muni_long['Sièges au CM'].notna()
    .groupby(t2_muni_long['Code commune'])
    .transform('sum')
)

In [ ]:
# Création % voix liste majo ----------------------------------
t2_muni_voix_majo = fprep.calcul_part_vote_liste_gagnante(t2_muni_long)
t2_muni_voix_majo[t2_muni_voix_majo['Code commune']=="75056"] # Vérification qu'on a ce qu'on veut pour Paris

In [ ]:
# Création indicateur 1 ----------------------------------
t2_muni_ind1 = fprep.crea_ind1(t2_muni_long, nomenclature_couleur_nuance)
print(t2_muni_ind1['ind1'].value_counts()) 

In [ ]:
# Création indicateur 2 ----------------------------------
t2_muni_ind2 = fprep.crea_ind2(t2_muni_long, nomenclature_couleur_nuance)
print(t2_muni_ind2['ind2'].value_counts())

In [ ]:
# Création indicateur 3 ----------------------------------
t2_muni_ind3 = fprep.crea_ind3(t2_muni_ind1.drop(columns=["ind1"])) 
t2_muni_ind3['ind3'].value_counts()

In [ ]:
# Création indicateur 4 ----------------------------------
t2_muni_ind4 = fprep.crea_ind4(t2_muni_ind1.drop(columns=["ind1"])) 
t2_muni_ind4['ind4'].value_counts() #aucune liste LHOR élue au deuxième tour

In [ ]:
# Table synthétique contenant les indicateurs --------
t2_muni_ind = (
    t2_muni_ind1
    .drop(columns=["Nuance_maj", "Couleur"])
    .merge(t2_muni_ind2[["Code commune", "ind2"]], on="Code commune", how="left")
    .merge(t2_muni_ind3[["Code commune", "ind3"]], on="Code commune", how="left")
    .merge(t2_muni_ind4[["Code commune", "ind4"]], on="Code commune", how="left")
    .merge(t2_muni_voix_majo[["Code commune", "part_votes_liste_gagnante", "Inscrits","Abstentions", "Exprimés"]], on="Code commune", how="left")
)

t2_muni_ind["tour"] = 2

t2_muni_ind.head()

In [ ]:
# Création de ind1
t1_muni_ind1 = fprep.crea_ind1(t1_muni_long, nomenclature_couleur_nuance)
print(t1_muni_ind1['ind1'].value_counts()) # beaucoup beaucoup de communes avec des listes sans étiquette ... 

In [ ]:
#Création de ind2
t1_muni_ind2 = fprep.crea_ind2(t1_muni_long, nomenclature_couleur_nuance)
print(t1_muni_ind2['ind2'].value_counts())

In [ ]:
#Création de ind3
t1_muni_ind3 = fprep.crea_ind3(t1_muni_ind1.drop(columns=["ind1"]))
print(t1_muni_ind3['ind3'].value_counts())

In [ ]:
# Création de ind4 
t1_muni_ind4 = fprep.crea_ind4(t1_muni_ind1.drop(columns=["ind1"])) 
t1_muni_ind4['ind4'].value_counts() 

In [ ]:
# Jointure des tables d'indicateurs + ajout indicatrice tour 1 ou 2

t1_muni_ind = (
    t1_muni_ind1
    .drop(columns=["Nuance_maj", "Couleur"])
    .merge(t1_muni_ind2[["Code commune", "ind2"]], on="Code commune", how="left")
    .merge(t1_muni_ind3[["Code commune", "ind3"]], on="Code commune", how="left")
    .merge(t1_muni_ind4[["Code commune", "ind4"]], on="Code commune", how="left")
    .merge(t1_muni_voix_majo[["Code commune", "part_votes_liste_gagnante", "Inscrits","Abstentions", "Exprimés"]], on="Code commune", how="left")
)

t1_muni_ind["tour"] = 1

t1_muni_ind.head()

### Empiler les tables des deux tours

In [ ]:
table_complete_muni = pd.concat([t1_muni_ind, t2_muni_ind], ignore_index=True)
table_complete_muni.head

In [ ]:
pd.crosstab(table_complete_muni['ind1'], table_complete_muni['ind2']) #coincident parfaitement 

In [ ]:
table_complete_muni['ind3'].value_counts() #faible effectif de ind3, analyse compromise 

In [ ]:
table_complete_muni['ind4'].value_counts() #presque pas d'effectif de ind4, ne sera pas analysé

In [ ]:
pd.crosstab(table_complete_muni['ind1'], table_complete_muni['Nombre candidats'])

## Jointure des tables sur code geo 

In [ ]:
table_analyse = filosofi.merge(bistrots_par_ville, left_on="CODGEO", right_on="code_commune", how="left")
print(filosofi.shape[0])
print(bistrots_par_ville["code_commune"].nunique())
print(bistrots_par_ville.shape[0])
print(table_analyse.shape[0])

In [ ]:
table_analyse_muni = table_complete_muni.merge(table_analyse, left_on="Code commune", right_on="CODGEO", how="left")
print(table_analyse.shape[0])
print(table_analyse["CODGEO"].nunique())
print(table_complete_muni["Code commune"].nunique())
print(table_complete_muni.shape[0])
print(table_analyse_muni.shape[0])

In [ ]:
table_analyse_muni["nb_etablissements"] = table_analyse_muni["nb_etablissements"].fillna(0).astype(int)

In [ ]:
FILE_KEY_OUT_S3 = "table_analyse_muni.csv"
FILE_PATH_OUT_S3 = BUCKET + "/" + FILE_KEY_OUT_S3

with fs.open(FILE_PATH_OUT_S3, 'w') as file_out:
    table_analyse_muni.to_csv(file_out)

In [ ]:
import os

print(os.getcwd())
table_analyse_muni.to_pickle("table_analyse_muni.pkl")
